# SOXS Feature and Constrained Defense Ablation

## 假设与边界

检验两件不同的事：SOXS路径/跟踪偏差是否能作为SOXL相对SOXX择时特征；受约束SOXS短期仓位是否能改善尾部风险。默认仓位是SOXX，SOXL与SOXS互斥，SOXS上限20%、连续最多五日。候选参数只在2017–2019 tune，2020–2022 validation用于模型组选择，2023–2026 outer只诊断。

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Markdown, display

HERE = Path.cwd().resolve()
ROOT = next(path for path in (HERE, *HERE.parents) if (path / 'src').is_dir() and (path / 'SPEC.md').exists())
if ROOT.as_posix() not in sys.path:
    sys.path.insert(0, ROOT.as_posix())

import qlib
from qlib.config import REG_US
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R

PROVIDER = ROOT / 'data' / 'qlib' / 'soxl_soxx_soxs'
qlib.init(provider_uri=PROVIDER.as_posix(), region=REG_US, kernels=1)
calendar = D.calendar(start_time='2010-03-11', end_time='2026-07-21', freq='day')

from src.workflow.soxs_defensive_hurdle import run_soxs_defensive_hurdle_card

CARD = ROOT / 'research' / 'soxs_defensive_hurdle_v1'
summary = run_soxs_defensive_hurdle_card(root=ROOT)
display({
    'calendar_rows': len(calendar),
    'as_of': summary['data_audit']['as_of'],
    'recorder_id': summary['recorder_id'],
    'promotion_decision': summary['promotion_decision'],
    'selection_uses_outer_test': summary['selection_uses_outer_test'],
})

[65756:MainThread](2026-07-22 14:54:46,075) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[65756:MainThread](2026-07-22 14:54:46,077) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[65756:MainThread](2026-07-22 14:54:46,077) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_soxx_soxs')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[65756:MainThread](2026-07-22 14:54:47,025) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[65756:MainThread](2026-07-22 14:54:47,054) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[65756:MainThread](2026-07-22 14:54:52,889) INFO - qlib.workflow - [exp.py:258] - Experiment 868189153558588533 starts running ...


[65756:MainThread](2026-07-22 14:54:52,949) INFO - qlib.workflow - [recorder.py:345] - Recorder 7180e35707854a4aab1b1a8f996f17f5 starts running under Experiment 868189153558588533 ...


[65756:MainThread](2026-07-22 14:54:53,134) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'calendar_rows': 4115,
 'as_of': '2026-07-21',
 'recorder_id': '7180e35707854a4aab1b1a8f996f17f5',
 'promotion_decision': 'research_reject_soxs_ablation_failed_validation_gates',
 'selection_uses_outer_test': False}

## Tune固定参数与Validation消融

In [2]:
selection = pd.read_csv(CARD / summary['sidecars']['model_selection'])
comparison = pd.read_csv(CARD / summary['sidecars']['backtest_benchmark_comparison'])
display(selection[[
    'variant', 'attack_threshold', 'attack_tilt', 'defense_threshold', 'soxs_weight',
    'validation_ratio_10bps', 'validation_ratio_25bps',
    'validation_active_ir_10bps', 'validation_max_drawdown_10bps',
    'validation_bootstrap_outperformance_probability_10bps', 'validation_gate_pass',
]])
display(comparison.loc[comparison['cost_bps'].eq(10), [
    'segment', 'variant', 'return_pct', 'terminal_wealth_ratio_vs_soxx',
    'active_information_ratio', 'max_drawdown_pct', 'worst_day_pct',
    'cvar5_pct', 'worst_rolling_5d_pct', 'avg_soxs_position_pct',
    'avg_cash_position_pct', 'upside_capture_pct',
]])

,variant,attack_threshold,attack_tilt,defense_threshold,soxs_weight,validation_ratio_10bps,validation_ratio_25bps,validation_active_ir_10bps,validation_max_drawdown_10bps,validation_bootstrap_outperformance_probability_10bps,validation_gate_pass
0,soxs_defense_only,1.00,0.0,0.6,0.2,0.941661,0.875791,-0.178297,-23.110845,0.446289,False
1,soxs_features_and_defense,0.60,0.2,0.6,0.2,0.894258,0.827630,-0.218099,-25.858784,0.392822,False
2,soxs_features_no_trade,0.60,0.2,1.0,0.0,0.861634,0.840509,-0.460157,-49.127889,0.251465,False
3,base_no_soxs,0.55,0.2,1.0,0.0,0.828848,0.811577,-0.125893,-59.088484,0.194580,False


,segment,variant,return_pct,terminal_wealth_ratio_vs_soxx,active_information_ratio,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,avg_soxs_position_pct,avg_cash_position_pct,upside_capture_pct
0,validation,soxx_buyhold,40.097364,1.000000,0.000000,-47.374011,-9.325051,-5.158647,-17.254933,0.000000,0.000000,33.758318
1,validation,base_no_soxs,16.119395,0.828848,-0.125893,-59.088484,-12.955748,-6.875221,-26.706279,0.000000,0.000000,41.823832
2,validation,soxs_features_no_trade,20.712615,0.861634,-0.460157,-49.127889,-12.955748,-5.914889,-26.706279,0.000000,0.000000,36.204805
3,validation,soxs_defense_only,31.924167,0.941661,-0.178297,-23.110845,-6.879413,-3.635861,-14.465743,2.307692,47.427056,10.853485
4,validation,soxs_features_and_defense,25.283232,0.894258,-0.218099,-25.858784,-9.574467,-3.888885,-18.458072,2.307692,47.427056,11.188877
15,outer,soxx_buyhold,379.937530,1.000000,0.000000,-42.790391,-9.654278,-5.179796,-17.688154,0.000000,0.000000,33.752633
16,outer,base_no_soxs,450.376416,1.146767,0.685943,-48.357568,-13.188470,-6.439813,-23.023214,0.000000,0.000000,40.216531
17,outer,soxs_features_no_trade,443.695999,1.132847,0.665826,-46.586831,-13.188470,-5.993889,-23.023214,0.000000,0.000000,38.079416
18,outer,soxs_defense_only,85.169455,0.385820,-0.948163,-30.366398,-6.996489,-3.267120,-11.475402,2.705750,48.365276,12.103835
19,outer,soxs_features_and_defense,86.447944,0.388484,-0.932098,-31.164826,-6.996489,-3.444664,-13.869517,2.705750,48.365276,12.801889


## SOXS特征是否提供增量

In [3]:
importance = pd.read_csv(CARD / summary['sidecars']['feature_importance'])
bootstrap = pd.read_csv(CARD / summary['sidecars']['bootstrap_metrics'])
display(importance.groupby('model', sort=False).head(5))
display(bootstrap.loc[bootstrap['cost_bps'].eq(10)])

,model,feature,coefficient,abs_coefficient
0,attack_base,soxx_drawdown60,0.367219,0.367219
1,attack_base,soxx_vol20,0.279622,0.279622
2,attack_base,soxx_trend60,-0.145183,0.145183
3,attack_base,soxx_trend20,-0.136255,0.136255
4,attack_base,relative_momentum20,-0.108155,0.108155
5,attack_soxs,soxx_vol20,0.424814,0.424814
6,attack_soxs,soxx_drawdown60,0.316100,0.316100
7,attack_soxs,inverse_tracking_gap20,-0.177193,0.177193
8,attack_soxs,soxx_trend60,-0.144426,0.144426
9,attack_soxs,soxl_soxs_path20,0.126276,0.126276


,variant,cost_bps,path_count,mean_terminal_wealth_ratio,median_terminal_wealth_ratio,outperformance_probability,active_information_ratio,terminal_ratio_p05,terminal_ratio_p10,terminal_ratio_p90,terminal_ratio_p95
0,base_no_soxs,10,4096,0.844802,0.829925,0.194580,-0.894580,0.576450,0.622108,1.087778,1.159583
1,soxs_features_no_trade,10,4096,0.876482,0.873458,0.251465,-0.740043,0.592375,0.649061,1.101024,1.157453
2,soxs_defense_only,10,4096,1.028007,0.940896,0.446289,-0.142104,0.473879,0.552741,1.607641,1.865405
3,soxs_features_and_defense,10,4096,0.971651,0.890264,0.392822,-0.275163,0.454188,0.530814,1.503403,1.738424


## 最终中文结论

In [4]:
recorder = R.get_recorder(recorder_id=summary['recorder_id'], experiment_name=summary['experiment'])
display({
    'recorder_local_dir': recorder.get_local_dir(),
    'recorder_only_artifacts': summary['artifact_policy']['recorder_only_artifacts'],
})
display(Markdown((CARD / 'conclusion.md').read_text(encoding='utf-8')))

{'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/soxs_defensive_hurdle_v1/qlib_recorder/868189153558588533/7180e35707854a4aab1b1a8f996f17f5',
 'recorder_only_artifacts': ['artifacts/soxs_defensive_hurdle_v1/models/',
  'artifacts/soxs_defensive_hurdle_v1/probabilities.csv',
  'artifacts/soxs_defensive_hurdle_v1/positions.csv',
  'artifacts/soxs_defensive_hurdle_v1/strategy_returns.csv',
  'artifacts/soxs_defensive_hurdle_v1/bootstrap_terminal_wealth.csv']}

# 中文结论

## 跑了什么

从独立 qlib SOXL/SOXX/SOXS provider 读取复权后的 open/close，比较基础特征、仅增加 SOXS 特征、仅允许受约束 SOXS 防守、SOXS 特征加防守四组消融。候选参数只在 2017–2019 tune 选择，2011–2019-11 refit，2020–2022 validation，2023–2026-07 outer 只诊断。SOXL 与 SOXS 互斥，SOXS 上限 20%，连续最多五日，之后转现金直到风险信号复位。

## Validation

选中 `soxs_defense_only`。10bp：相对 SOXX 财富比 0.9417，Active IR -0.1783，收益 31.92%，MaxDD -23.11%，SOXS 平均仓位 2.31%。validation gate=False。

## Outer诊断

10bp：收益 85.17%，SOXX 379.94%，财富比 0.3858，MaxDD -30.37%。outer 不参与选择。

## 决定

`research_reject_soxs_ablation_failed_validation_gates`。SOXS 是否提供增量必须以 feature-only 与 tradable-sleeve 的差异判断；已观察 outer 不能用于返调阈值。下一步只能进入 sealed-forward paper shadow，或在 validation 失败时停止本架构。

不构成买卖建议。
